# NB1: Baseline Multi-Agent Collaboration

**2-minute intro script:** In a managed software-delivery team, a manager delegates to a Requirements Analyst, Implementation Planner, Delivery Coordinator, and Risk Reviewer. It feels powerful because context flows from one role to the next. This notebook recreates that collaboration without requiring live CrewAI or API keys. The production lesson is simple: before you add memory, schemas, MCP, or self-repair, learners must first see the handoff. One agent creates context, another consumes it, and a manager coordinates the sequence.

In [ ]:
from dataclasses import dataclass

@dataclass
class DeliveryAgent:
    role: str
    goal: str

    def run(self, task: str, context: str = "") -> str:
        if self.role == "Requirements Analyst":
            return (
                "Requirement insight: the authentication API must support "
                "login, logout, token refresh, and rate limiting."
            )
        if self.role == "Implementation Planner":
            return (
                f"Implementation plan based on [{context}]: build REST "
                "endpoints, store tokens securely, and add unit tests."
            )
        if self.role == "Delivery Coordinator":
            return (
                f"Delivery plan based on [{context}]: create a pull request, "
                "run CI checks, and prepare reviewer notes."
            )
        if self.role == "Risk Reviewer":
            return (
                f"Risk review based on [{context}]: require security notes, "
                "rate-limit tests, and rollback instructions before release."
            )
        raise ValueError(f"Unknown role: {self.role}")

In [ ]:
agents = [
    DeliveryAgent("Requirements Analyst", "Clarify product requirements"),
    DeliveryAgent("Implementation Planner", "Plan the technical approach"),
    DeliveryAgent("Delivery Coordinator", "Prepare release execution"),
    DeliveryAgent("Risk Reviewer", "Assess release risk"),
]

context = ""
transcript = []
for agent in agents:
    output = agent.run("Prepare an internal authentication API release.", context)
    transcript.append((agent.role, output))
    context = output

for role, output in transcript:
    print(f"=== {role} ===")
    print(output)
    print()

## From Static Agents to Mock LLM Output

The first cells are a deterministic shell: useful for learning the collaboration pattern without API keys. Now we introduce the missing brain. A mock LLM returns messy JSON the way a real model sometimes does: invalid enum values, out-of-range confidence, and hallucinated fields. The Pydantic contract catches the chaos before the next agent consumes it.

In [ ]:
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field, ValidationError

class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")

class DeliveryHandoff(StrictModel):
    role: Literal[
        "Requirements Analyst",
        "Implementation Planner",
        "Delivery Coordinator",
        "Risk Reviewer",
    ]
    summary: str = Field(min_length=20)
    confidence: float = Field(ge=0.0, le=1.0)
    next_agent_hint: Literal[
        "Implementation Planner",
        "Delivery Coordinator",
        "Risk Reviewer",
        "Done",
    ]

def mock_delivery_llm(role: str, attempt: int = 0) -> dict:
    """Simulate a nondeterministic LLM response without API keys."""
    if attempt == 0:
        return {
            "role": role,
            "summary": "Looks good",  # too short for production handoff
            "confidence": 1.4,  # invalid confidence range
            "next_agent_hint": "Maybe Compliance?",  # invalid enum
            "hallucinated_field": "The model invented this field.",
        }
    return {
        "role": role,
        "summary": "The authentication API release needs rate limiting, security notes, and rollback instructions.",
        "confidence": 0.78,
        "next_agent_hint": "Implementation Planner",
    }

print("--- Attempt 1: mock LLM hallucinates ---")
try:
    DeliveryHandoff.model_validate(mock_delivery_llm("Requirements Analyst", attempt=0))
except ValidationError as exc:
    print("Pydantic rejected the handoff:")
    for error in exc.errors():
        print(f"- {error['loc']}: {error['msg']}")
    print("Repair feedback goes back to the agent.\n")

print("--- Attempt 2: mock LLM repairs ---")
valid_handoff = DeliveryHandoff.model_validate(
    mock_delivery_llm("Requirements Analyst", attempt=1)
)
print(valid_handoff.model_dump_json(indent=2))

In [ ]:
# ==========================================
# LIVE LLM EXECUTION (Optional)
# ==========================================
# The cells above run offline using deterministic mocks.
# To see a real LLM generate output constrained by this schema:
#
#   pip install openai instructor
#   export OPENAI_API_KEY="..."
#
# Keep this False for workshops unless learners have API keys.
USE_LIVE_LLM = False

if USE_LIVE_LLM:
    import os
    import instructor
    from openai import OpenAI

    client = instructor.from_openai(OpenAI(api_key=os.environ["OPENAI_API_KEY"]))
    model_name = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")

    live_result = client.chat.completions.create(
        model=model_name,
        response_model=DeliveryHandoff,
        messages=[
            {
                "role": "user",
                "content": 'Return a structured software-delivery handoff for an authentication API release. Include role, summary, confidence, and next_agent_hint.',
            }
        ],
    )
    print(live_result.model_dump_json(indent=2))

## Production Mapping

CrewAI can run this as `Process.hierarchical`; AutoGen can run it as a group chat; LangGraph can model it as a state graph. The collaboration pattern is the same: manager, specialists, handoffs, and review.

## 🧪 Exercises: Building Your First Agent Team

**The Story:** Imagine your team is preparing an internal authentication API release. The implementation looks fine, but the Risk Reviewer forgets rate limiting and rollback instructions. The release may work in a demo, but it is not production-ready. Right now, your agents are just passing text. How do we ensure the final handoff is governable?

**Your Mission:**
1. **The Release Gate:** Add a `Compliance Reviewer` agent after the Risk Reviewer. Its goal is to confirm the release plan includes security notes, rate-limit tests, and rollback instructions.
2. **The Scope Change:** Change the task from `authentication API` to another internal software request such as `support ticket API` or `document upload service`. Notice which handoffs should adapt.
3. **The Black Box Problem:** In production, if an agent approves a risky release, we need to know why. Record each handoff in a list called `audit_log` that captures the `role`, `input_context`, and `output`.
4. **The Whisper Down the Lane:** Identify one place where context could fade or be misread as it passes from Requirements Analyst to Risk Reviewer. How would you fix it?

### Builder Exercise: The Governed Logistics Weather Agent

**The Analogy:** A logistics coordinator should not carry the master key to the weather API. It should ask a concierge. The concierge checks the badge, calls the API, and returns only a validated shipping-risk receipt.

**Semantic Building Blocks:**
- `AgentIdentity`: who is asking?
- `WeatherRequest`: what exact input is allowed?
- `GovernedMCPGateway`: what boundary hides the API key and enforces scope?
- `WeatherResponse`: what typed result may cross back to the agent?

**Your Mission:**
1. Create a `logistics_coordinator` identity that can call `get_weather`.
2. Create an `external_widget` identity that cannot call `get_weather` or `cancel_shipment`.
3. Define strict Pydantic contracts for `WeatherRequest(city, units)` and `WeatherResponse(city, temperature, condition, shipping_risk)`.
4. Route all weather access through a gateway. The API key must live inside the gateway, never inside the agent prompt or identity.
5. Prove the authorized agent receives a typed `WeatherResponse`, the unauthorized widget raises `PermissionError`, and a hallucinated field fails schema validation.

**Production Check:** If the LLM adds `vibes="sunny enough"` or requests an unsupported unit, Pydantic should reject it before downstream agents act.

**The Takeaway:** Multi-agent collaboration is powerful, but without an audit trail and compliance checks, it's just a black box generating text.